# Example 4: LaTeX Report Generation

Demonstrates automatic generation of professional LaTeX documents with results.

In [ ]:
import numpy as np
import os
from mathdoc import DataLoader, Approximator, ErrorAnalyzer, LatexReport, Visualizer

## 4.1 Prepare Data and Approximation

In [ ]:
# Generate data
loader = DataLoader(random_seed=42)
x, y = loader.load_synthetic(n_points=50, noise=0.05, function='sine')

# Fit approximation
approx = Approximator(x, y, basis_type='chebyshev', degree=10)
coeffs = approx.fit()
y_approx = approx.evaluate(x)

# Error analysis
error_analyzer = ErrorAnalyzer(y, y_approx)
errors = error_analyzer.summary(y_approx)

print('Approximation Results:')
print(f'Number of data points: {len(x)}')
print(f'Basis type: Chebyshev')
print(f'Polynomial degree: 10')
print(f'L2 error norm: {errors["l2_norm"]:.8f}')
print(f'Max error: {errors["l_inf_norm"]:.8f}')
print(f'Relative error: {errors["relative_error"]:.8f}')
print(f'MSE: {errors["mse"]:.8f}')

## 4.2 Create LaTeX Report

In [ ]:
# Initialize report
report = LatexReport(
    title='Polynomial Approximation Analysis',
    author='MathDoc Educational Project'
)

# Add introduction
report.add_section('Introduction')
report.add_text(
    'This report presents the results of polynomial approximation using Chebyshev basis. '
    'The goal is to fit a smooth curve through noisy data points using least squares approximation.'
)

# Add methodology section
report.add_section('Methodology')
report.add_subsection('Basis Functions')
report.add_text(
    'We use Chebyshev polynomials of the first kind as basis functions. '
    'These polynomials are optimal for polynomial approximation on intervals.'
)

report.add_math(
    r'T_0(x) = 1, \quad T_1(x) = x, \quad T_{n+1}(x) = 2x T_n(x) - T_{n-1}(x)',
    display=True
)

report.add_subsection('Least Squares Problem')
report.add_text('We solve the least squares problem:')
report.add_math(
    r'\min_c \|Ac - y\|_2^2',
    display=True
)

report.add_text('where $A$ is the matrix of basis functions and $c$ are the coefficients.')

## 4.3 Add Results Section

In [ ]:
# Results section
report.add_section('Results')
report.add_subsection('Approximation Parameters')

# Parameters table
params = np.array([
    [len(x), len(coeffs), coeffs[0]],
    [np.min(x), np.max(x), 0],
    [np.min(y), np.max(y), 0]
])

report.add_table(
    params,
    headers=['Number of Data Points', 'Number of Coefficients', 'First Coefficient'],
    caption='Approximation Parameters'
)

report.add_subsection('Error Metrics')

# Error metrics table
error_data = np.array([
    [errors['l2_norm'], errors['l_inf_norm'], errors['relative_error']],
    [errors['mse'], 0, 0]
])

report.add_table(
    error_data[:1],
    headers=[r'$\|e\|_2$', r'$\|e\|_\infty$', 'Relative Error'],
    caption='Error Metrics'
)

# Mathematical result
report.add_equation_box('error_norm', f'\|e\|_2 = {errors["l2_norm"]:.6e}')
report.add_equation_box('max_error', f'\max_i |e_i| = {errors["l_inf_norm"]:.6e}')
report.add_equation_box('mse', f'MSE = {errors["mse"]:.6e}')

## 4.4 Add Visualization

In [ ]:
# Create visualizations
viz = Visualizer()

# Dense evaluation for smooth curves
x_dense = np.linspace(np.min(x), np.max(x), 300)
y_dense = approx.evaluate(x_dense)

# Approximation plot
viz.plot_approximation(
    x, y, y_approx,
    title='Chebyshev Approximation (Degree 10)',
    filename='approx_result.png'
)

print('Visualization saved: approx_result.png')

## 4.5 Add Convergence Analysis

In [ ]:
# Convergence analysis
report.add_section('Convergence Analysis')
report.add_text('We examine how the approximation error decreases with increasing polynomial degree.')

# Compute errors for different degrees
degrees = np.arange(1, 15)
conv_errors = []

for d in degrees:
    app = Approximator(x, y, basis_type='chebyshev', degree=d)
    app.fit()
    y_a = app.evaluate(x)
    ea = ErrorAnalyzer(y, y_a)
    conv_errors.append(ea.l2_norm(y_a))

# Plot convergence
viz.plot_convergence(
    list(degrees),
    conv_errors,
    title='Convergence with Polynomial Degree',
    filename='convergence.png'
)

report.add_figure('convergence.png', 'Convergence of approximation error with degree')

## 4.6 Add Conclusions

In [ ]:
# Conclusions
report.add_section('Conclusions')
report.add_text(
    f'Using a Chebyshev polynomial of degree 10, we achieved an approximation with '
    f'L2 error norm of {errors["l2_norm"]:.2e}. '
    f'The maximum pointwise error is {errors["l_inf_norm"]:.2e}. '
    f'This demonstrates the effectiveness of orthogonal polynomial bases for function approximation.'
)

# References
report.add_section('References')
report.add_text(
    r'\begin{enumerate}'
    r'\item Boyd, J. P. (2001). \textit{Chebyshev and Fourier Spectral Methods}. Dover Publications.'
    r'\item Trefethen, L. N. (2013). \textit{Approximation Theory and Approximation Practice}. SIAM.'
    r'\end{enumerate}'
)

## 4.7 Generate LaTeX File

In [ ]:
# Generate LaTeX document
output_file = 'analysis_report.tex'
report.generate(output_file)

print(f'\n=== LaTeX Report Generated ===')
print(f'File: {output_file}')
print(f'Size: {os.path.getsize(output_file)} bytes')
print(f'\nTo compile the report, run:')
print(f'  pdflatex {output_file}')
print(f'  pdflatex {output_file}  # Run twice for TOC')

## 4.8 Display Report Content

In [ ]:
# Show first part of generated LaTeX
with open(output_file, 'r') as f:
    content = f.read()
    # Show first 2000 characters
    print('First 1500 characters of generated LaTeX:')
    print('='*60)
    print(content[:1500])
    print('...')
    print('='*60)

## Summary

This example demonstrated:
1. **Data Approximation**: Fitting a Chebyshev polynomial to noisy data
2. **Error Analysis**: Computing and interpreting various error metrics
3. **Visualization**: Creating publication-ready plots
4. **LaTeX Export**: Automatically generating a professional technical report

The generated LaTeX file can be compiled to PDF using pdflatex and includes:
- Mathematical equations with proper formatting
- Formatted tables with results
- Embedded figures
- Professional document structure with table of contents